<a href="https://colab.research.google.com/github/jamalinu/amazigh-nlp-spacy/blob/main/Building_Custom_NLP_Pipelines_for_Low_Resource_Languages_The_Case_of_Amazigh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import spacy
from spacy.symbols import ORTH

# Usamos 'ber' para representar las lenguas bereberes (Amazigh)
# Si 'ber' da error por no estar registrado, usamos 'xx' (multi-language)
# o simplemente 'blank' con un código de idioma nuevo.
try:
    nlp = spacy.blank("ber")
except ImportError:
    nlp = spacy.blank("xx") # Fallback a multi-idioma si 'ber' no está en la base

# Definimos una frase de ejemplo (puedes usar Tifinagh o Latino)
text = "Akal n Imazighen"

# --- Aquí es donde aplicas tu conocimiento filológico ---
# Vamos a enseñarle al tokenizador que 'n' es una unidad independiente
special_case = [{ORTH: "n"}]
nlp.tokenizer.add_special_case("n", special_case)

doc = nlp(text)

print(f"Tokens detectados: {[token.text for token in doc]}")

Tokens detectados: ['Akal', 'n', 'Imazighen']


In [5]:
# Definimos una frase con un pronombre pegado (clítico)
text_clitic = "fkan-as akal" # Le dieron tierra

# En un pipeline profesional, no queremos que 'fkan-as' sea un solo token.
# Queremos: ['fkan', '-', 'as', 'akal']

# Añadimos la regla de excepción
special_case_clitic = [
    {ORTH: "fkan", "LEMMA": "fki", "POS": "VERB"},
    {ORTH: "-"},
    {ORTH: "as", "POS": "PRON"}
]
nlp.tokenizer.add_special_case("fkan-as", special_case_clitic)

doc = nlp(text_clitic)

print("Segmentación avanzada:")
for token in doc:
    print(f"Token: {token.text:10} | Lema: {token.lemma_:10} | Categoría: {token.pos_}")

ValueError: [E1005] Unable to set attribute 'LEMMA' in tokenizer exception for 'fkan-as'. Tokenizer exceptions are only allowed to specify ORTH and NORM.

In [7]:
import spacy
from spacy.symbols import ORTH

# Usamos "xx" (Multi-language) como base porque "ber" no existe en la librería estándar todavía
nlp = spacy.blank("xx")

# 1. Definimos la excepción (Clítico)
# Usaremos el ejemplo: "fkan-as" -> "fkan" (dieron) + "-" + "as" (le)
special_case_clitic = [
    {ORTH: "fkan"},
    {ORTH: "-"},
    {ORTH: "as"}
]

# 2. Añadimos la regla al tokenizer
# Esto le dice a spaCy: "Cuando veas esta cadena exacta, no la trates como una palabra, sino como tres"
nlp.tokenizer.add_special_case("fkan-as", special_case_clitic)

# 3. Procesamos una frase de ejemplo
text_clitic = "fkan-as akal"
doc = nlp(text_clitic)

# 4. Verificamos la segmentación
print("Tokens segmentados en Amazigh:")
print([token.text for token in doc])

Tokens segmentados en Amazigh:
['fkan', '-', 'as', 'akal']


In [8]:
# Función de normalización simple (puedes ampliarla según tu conocimiento filológico)
def normalize_amazigh(text):
    # Ejemplo: Asegurar que el carácter de separación sea siempre el mismo
    # O corregir variantes de letras Tifinagh comunes
    text = text.replace("ⵗ", "ⵖ") # Ejemplo de normalización de variantes de 'G'
    return text

raw_text = "akal n ⵉⵎⴰⵣⵉⵖⵏ"
clean_text = normalize_amazigh(raw_text)
doc = nlp(clean_text)

print(f"Texto normalizado y tokenizado: {[t.text for t in doc]}")

Texto normalizado y tokenizado: ['akal', 'n', 'ⵉⵎⴰⵣⵉⵖⵏ']
